In [14]:
import optuna
import numpy as np
import pandas as pd
from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score
from xgboost import XGBClassifier

In [15]:
wine = load_wine()

X = pd.DataFrame(wine.data, columns=wine.feature_names)
y = wine.target

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [21]:
print(wine.target_names)

['class_0' 'class_1' 'class_2']


In [16]:
X_train

,alcohol,malic_acid,ash,alcalinity_of_ash,magnesium,total_phenols,flavanoids,nonflavanoid_phenols,proanthocyanins,color_intensity,hue,od280/od315_of_diluted_wines,proline
158,14.34,1.68,2.70,25.0,98.0,2.80,1.31,0.53,2.70,13.00,0.57,1.96,660.0
137,12.53,5.51,2.64,25.0,96.0,1.79,0.60,0.63,1.10,5.00,0.82,1.69,515.0
98,12.37,1.07,2.10,18.5,88.0,3.52,3.75,0.24,1.95,4.50,1.04,2.77,660.0
159,13.48,1.67,2.64,22.5,89.0,2.60,1.10,0.52,2.29,11.75,0.57,1.78,620.0
38,13.07,1.50,2.10,15.5,98.0,2.40,2.64,0.28,1.37,3.70,1.18,2.69,1020.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
71,13.86,1.51,2.67,25.0,86.0,2.95,2.86,0.21,1.87,3.38,1.36,3.16,410.0
106,12.25,1.73,2.12,19.0,80.0,1.65,2.03,0.37,1.63,3.40,1.00,3.17,510.0
14,14.38,1.87,2.38,12.0,102.0,3.30,3.64,0.29,2.96,7.50,1.20,3.00,1547.0
92,12.69,1.53,2.26,20.7,80.0,1.38,1.46,0.58,1.62,3.05,0.96,2.06,495.0


In [17]:
y_train

array([2, 2, 1, 2, 0, 1, 1, 1, 2, 0, 1, 1, 2, 0, 1, 0, 0, 2, 2, 1, 1, 0,
       1, 0, 2, 1, 1, 2, 0, 0, 0, 2, 0, 0, 1, 2, 1, 0, 2, 1, 0, 2, 1, 1,
       0, 1, 0, 0, 1, 0, 0, 2, 1, 1, 1, 0, 1, 1, 1, 2, 2, 0, 1, 2, 2, 1,
       1, 0, 1, 2, 2, 1, 2, 1, 1, 1, 0, 0, 2, 0, 2, 0, 0, 1, 1, 0, 0, 0,
       1, 0, 1, 2, 1, 1, 1, 2, 2, 1, 0, 0, 1, 2, 2, 0, 1, 2, 2, 2, 2, 1,
       0, 1, 0, 2, 0, 0, 1, 0, 0, 2, 1, 0, 2, 2, 0, 0, 2, 2, 2, 1, 1, 1,
       1, 1, 1, 2, 0, 1, 1, 0, 1, 1])

In [18]:
def objective(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 50, 200),
        'max_depth': trial.suggest_int('max_depth', 2, 8),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'random_state': 42,
        'eval_metric': 'mlogloss'
    }

    model = XGBClassifier(**params)
    model.fit(X_train, y_train)

    preds = model.predict(X_test)
    score = f1_score(y_test, preds, average='macro')

    return score

In [20]:
study = optuna.create_study(direction='maximize')

study.optimize(objective, n_trials=200)

print("Best F1 Score: ", study.best_value)
print("Best Parameters: ", study.best_params)

[I 2026-10-06 11:37:49,159] A new study created in memory with name: no-name-44f865a8-eded-49b1-b9b0-1b02bef6a331
[I 2026-10-06 11:37:49,241] Trial 0 finished with value: 1.0 and parameters: {'n_estimators': 147, 'max_depth': 2, 'learning_rate': 0.2896368873660441, 'subsample': 0.9210133973637261, 'colsample_bytree': 0.9712574682101065}. Best is trial 0 with value: 1.0.
[I 2026-10-06 11:37:49,308] Trial 1 finished with value: 1.0 and parameters: {'n_estimators': 111, 'max_depth': 5, 'learning_rate': 0.22849394622698213, 'subsample': 0.8404972444733474, 'colsample_bytree': 0.9398306381725838}. Best is trial 0 with value: 1.0.
[I 2026-10-06 11:37:49,359] Trial 2 finished with value: 1.0 and parameters: {'n_estimators': 63, 'max_depth': 2, 'learning_rate': 0.029927349756728505, 'subsample': 0.6800420583453597, 'colsample_bytree': 0.7985061168841703}. Best is trial 0 with value: 1.0.
[I 2026-10-06 11:37:49,445] Trial 3 finished with value: 1.0 and parameters: {'n_estimators': 120, 'max_dep

Best F1 Score:  1.0
Best Parameters:  {'n_estimators': 147, 'max_depth': 2, 'learning_rate': 0.2896368873660441, 'subsample': 0.9210133973637261, 'colsample_bytree': 0.9712574682101065}


In [26]:
est_xgb = XGBClassifier(**study.best_params, eval_metric='mlogloss', random_state=42)

In [27]:
est_xgb.fit(X_train, y_train)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=0.9712574682101065, device=None,
              early_stopping_rounds=None, enable_categorical=True,
              eval_metric='mlogloss', feature_types=None, feature_weights=None,
              gamma=None, grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=0.2896368873660441,
              max_bin=None, max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=2, max_leaves=None,
              min_child_weight=None, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=147, n_jobs=None,
              num_parallel_tree=None, ...)

In [28]:
preds = est_xgb.predict(X_test)
print(preds)

[0 0 2 0 1 0 1 2 1 2 0 2 0 1 0 1 1 1 0 1 0 1 1 2 2 2 1 1 1 0 0 1 2 0 0 0]
